In [ ]:
import warnings
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
train_valid_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"
test_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_테스트용(80개).csv"

# Optuna 설정부
n_trials = 200
# valid AUC에서 (train-valid gap)의 이 비율만큼 페널티를 줌
# gap이 0.10이면 0.10 * 0.5 = 0.05 감점 → 과적합 파라미터 억제
overfit_penalty = 0.5

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]


def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()
    mapped = lowered.map(
        {"1": 1, "0": 0, "y": 1, "n": 0, "yes": 1, "no": 0, "true": 1, "false": 0}
    )
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.where(mapped.notna(), numeric)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )
    print(target_dist)


def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])


def prepare_data(df, selected_features, numeric_features):
    df = df.copy()
    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["is_repurchase_num"] = to_binary(df["is_repurchase"])
    df = df[df["is_repurchase_num"].isin([0, 1])].copy()
    X = df[selected_features].copy()
    y = (df["is_repurchase_num"] == 0).astype(int)
    return X, y


# 데이터 로드부
train_valid_df = pd.read_csv(train_valid_file_path).copy()
test_df = pd.read_csv(test_file_path).copy()

selected_features = [col for col in train_valid_df.columns if col not in exclude_cols]
categorical_features = ["age_group"]
numeric_features = [col for col in selected_features if col not in categorical_features]

X_train_valid, y_train_valid = prepare_data(train_valid_df, selected_features, numeric_features)
X_test, y_test = prepare_data(test_df, selected_features, numeric_features)

print("분석 기준: 최종 파생 변수 + SVM Optuna 5-Fold 튜닝 (과적합 페널티 적용)")
print("양성 클래스 기준: is_repurchase == 0")
print(f"train/valid 데이터 수: {len(X_train_valid)}")
print(f"test 데이터 수: {len(X_test)}")
print(f"전체 데이터 수: {len(X_train_valid) + len(X_test)}")
print(f"train/valid 비율: {len(X_train_valid) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"test 비율: {len(X_test) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"train/valid 파일 전체 컬럼 수: {len(train_valid_df.columns)}")
print(f"test 파일 전체 컬럼 수: {len(test_df.columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print(f"과적합 페널티 가중치: {overfit_penalty}")
print("사용 변수")
print(selected_features)
print("train/valid 타깃 분포")
print_target_distribution(y_train_valid)
print("test 타깃 분포")
print_target_distribution(y_test)

if y_train_valid.nunique() < 2 or y_train_valid.value_counts().min() < 5:
    print("학습 불가: train/valid 타깃 클래스가 부족합니다.")
elif y_test.nunique() < 2:
    print("평가 불가: test 타깃 클래스가 부족합니다.")
else:
    def objective(trial):
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        params = {
            "C": trial.suggest_float("C", 0.001, 100.0, log=True),
            "kernel": kernel,
            "class_weight": "balanced",
            "random_state": 42,
            "cache_size": 1000,
        }
        if kernel == "rbf":
            params["gamma"] = trial.suggest_categorical("gamma", ["scale", "auto"])

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_train_scores = []
        fold_valid_scores = []

        for train_idx, valid_idx in skf.split(X_train_valid, y_train_valid):
            X_fold_train = X_train_valid.iloc[train_idx]
            X_fold_valid = X_train_valid.iloc[valid_idx]
            y_fold_train = y_train_valid.iloc[train_idx]
            y_fold_valid = y_train_valid.iloc[valid_idx]

            clf = Pipeline(steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", SVC(**params)),
            ])
            clf.fit(X_fold_train, y_fold_train)

            train_score = clf.decision_function(X_fold_train)
            valid_score = clf.decision_function(X_fold_valid)

            fold_train_scores.append(roc_auc_score(y_fold_train, train_score))
            fold_valid_scores.append(roc_auc_score(y_fold_valid, valid_score))

        mean_train = sum(fold_train_scores) / len(fold_train_scores)
        mean_valid = sum(fold_valid_scores) / len(fold_valid_scores)
        mean_gap = mean_train - mean_valid

        trial.set_user_attr("mean_valid_roc_auc", round(mean_valid, 4))
        trial.set_user_attr("mean_auc_gap", round(mean_gap, 4))

        # 과적합(train >> valid)이면 gap만큼 페널티를 주어 일반화 성능 위주로 탐색
        return mean_valid - overfit_penalty * max(0.0, mean_gap)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_mean_valid = best_trial.user_attrs.get("mean_valid_roc_auc", round(study.best_value, 4))
    best_mean_gap = best_trial.user_attrs.get("mean_auc_gap", None)

    print("Optuna best 조정 점수 (valid roc_auc - 과적합 페널티)")
    print(round(study.best_value, 4))
    print("Optuna best 5-fold valid roc_auc (페널티 적용 전)")
    print(best_mean_valid)
    print("Optuna best 5-fold train-valid auc gap")
    print(best_mean_gap)
    print("Optuna best params")
    print(study.best_params)

    best_params = {
        **study.best_params,
        "class_weight": "balanced",
        "random_state": 42,
        "cache_size": 1000,
    }

    print("최종 학습에 사용한 SVM 파라미터")
    print(best_params)

    final_clf = Pipeline(steps=[
        ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
        ("model", SVC(**best_params)),
    ])
    final_clf.fit(X_train_valid, y_train_valid)

    y_train_valid_pred = final_clf.predict(X_train_valid)
    y_test_pred = final_clf.predict(X_test)

    y_train_valid_score = final_clf.decision_function(X_train_valid)
    y_test_score = final_clf.decision_function(X_test)

    train_valid_roc_auc = roc_auc_score(y_train_valid, y_train_valid_score)
    test_roc_auc = roc_auc_score(y_test, y_test_score)
    auc_gap = train_valid_roc_auc - test_roc_auc

    result = {
        "model": "SVM_Optuna_5Fold",
        "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
        "train_valid_roc_auc": train_valid_roc_auc,
        "test_roc_auc": test_roc_auc,
        "auc_gap": auc_gap,
        "is_overfit": auc_gap >= 0.05,
    }

    results_df = (
        pd.DataFrame([result])
        .set_index("model")
        [["f1_score", "train_valid_roc_auc", "test_roc_auc", "auc_gap", "is_overfit"]]
        .round(4)
    )

    print("최종 테스트 성능")
    print(results_df.to_string())
